# Datafest 2026 

In [1]:
import pandas as pd
import dtale as dt
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import imageio
import os

In [2]:
df_enc = pd.read_csv('encounters.csv')
# pd.to_datetime('Date')
df_enc.rename(columns={'AdmissionType': 'Reason For Admission',
                       'AdmissionSource':'Prior To Admission',
                       'AdmitYear':'Year of Admission',
                       'AdmitMonth':  'Month of Admission',
                       'Date':'Date of Encounter',
                       'AttendingProviderDurableKey' : 'Attending Provider KEY',
                       'DepartmentKey' : 'Encounter - Department KEY',
                       'DischargeMonth': 'Month of Discharge',
                       'DischargeProviderDurableKey':'Last Provider to Patient (KEY TO PROVIDER)',
                       'DischargeYear':'Year of Discharge',
                       'EncounterKey': 'Index',
                       'IsEdVisit' : 'Emergency Visit(Yes/No)', 
                       'IsHospitalAdmission': 'Admission(Yes/No)',
                       'IsHospitalOutpatientVisit' : 'Out Patient(Yes/No)',
                       'IsInpatientAdmission': 'In Patient Admission(Yes/No)',
                       'IsObservation' : 'Under Obs.',
                       'IsOutpatientFaceToFaceVisit' : 'In Person Visit for Outpatient(Yes/No)',
                       'PatientDurableKey' : 'Encounter-Patient KEY',
                       'PrimaryDiagnosisKey': 'Encounter-Diagnosis KEY',
                       'ProviderDurableKey':'Encounter-Provider KEY',
                       'Type':'Type of Visit',
                       'VisitType':'Service Patient Received',
                       'VisitTypeDescription':'Visit Type Description'}, inplace = True)

df_enc.set_index('Index', inplace = True)
df_enc['Date of Encounter'] = pd.to_datetime(df_enc['Date of Encounter'], format = '%m/%d/%y') #changing format of the date column

df_enc.drop([ 'AdmitDay','AdmitHour',
'AdmitMinute','DischargeDay','DischargeHour', 'DischargeMinute', 'Reason For Admission', 'Prior To Admission',
              'Year of Admission', 'Month of Admission', 'Date of Encounter', 'Month of Discharge', 'Year of Discharge', 
              'Emergency Visit(Yes/No)', 'Admission(Yes/No)', 'Out Patient(Yes/No)', 'In Patient Admission(Yes/No)',
              'Under Obs.', 'In Person Visit for Outpatient(Yes/No)', 'Type of Visit', 'Service Patient Received',
              'Visit Type Description'],axis = 1, inplace = True) # dropping columns

# df_enc['Year of Admission'] = df_enc['Year of Admission'].ffill() # Forward Filled all the Null rows
# df_enc['Month of Admission'] = df_enc['Date of Encounter'].dt.month # Filling month from Appointment

# df_enc['Year of Admission'].astype('int64')
# df_enc['Month of Admission'].astype('int64')

# df_enc['Year of Discharge'] = df_enc['Date of Encounter'].dt.year

# df_enc['IsHospitalAdmission'].filter(df_enc['IsHospitalAdmission'] == 1).count()
# type(df_enc['Date of Encounter'].iloc[0])

In [3]:
df_enc['AdmissionInstant'] = pd.to_datetime(df_enc['AdmissionInstant'], errors='coerce')
df_enc['temp_unix'] = pd.to_numeric(df_enc['AdmissionInstant']) / 10**9
df_enc.loc[df_enc['temp_unix'] < 0, 'temp_unix'] = np.nan
df_enc['temp_unix'] = df_enc['temp_unix'].interpolate(method='linear')
df_enc['AdmissionInstant'] = pd.to_datetime(df_enc['temp_unix'], unit='s')
df_enc.drop(columns=['temp_unix'], inplace=True)

In [4]:
df_enc['DischargeInstant'] = pd.to_datetime(df_enc['DischargeInstant'], errors='coerce')
df_enc['temp_unix'] = pd.to_numeric(df_enc['DischargeInstant']) / 10**9
df_enc.loc[df_enc['temp_unix'] < 0, 'temp_unix'] = np.nan
df_enc['temp_unix'] = df_enc['temp_unix'].interpolate(method='linear')
df_enc['DischargeInstant'] = pd.to_datetime(df_enc['temp_unix'], unit='s')
df_enc.drop(columns=['temp_unix'], inplace=True)

In [5]:
df_enc['Duration_Raw'] = df_enc['DischargeInstant'] - df_enc['AdmissionInstant']
df_enc['Duration_Hours'] = df_enc['Duration_Raw'] / pd.Timedelta(hours=1)

In [6]:
dt.show(df_enc.tail(1000))

In [7]:
# Filter for only human providers (where Attending Provider KEY is not -1)
# AND where Duration_Hours is greater than 0
human_providers_df = df_enc[(df_enc['Attending Provider KEY'] != -1) & 
                           (df_enc['Duration_Hours'] > 0)]

# Now you have the Duration_Hours for only human attendees with non-zero duration
human_duration_hours = human_providers_df['Duration_Hours']

# If you want statistics on this filtered data
avg_human_duration = human_duration_hours.mean()
max_human_duration = human_duration_hours.max()
min_human_duration = human_duration_hours.min()

# If you want to group by provider to see each human provider's times
provider_durations = human_providers_df.groupby('Attending Provider KEY')['Duration_Hours'].agg(['mean', 'sum', 'count'])

In [8]:
human_duration_hours.to_csv('HumanWorktime.csv')

In [9]:
# Filter for only robot/machine providers (where Attending Provider KEY is -1)
# AND where Duration_Hours is greater than 0
robot_providers_df = df_enc[(df_enc['Attending Provider KEY'] == -1) & 
                           (df_enc['Duration_Hours'] > 0)]

# Now you have the Duration_Hours for only robot attendees with non-zero duration
robot_duration_hours = robot_providers_df['Duration_Hours']

# If you want statistics on this filtered data
avg_robot_duration = robot_duration_hours.mean()
max_robot_duration = robot_duration_hours.max()
min_robot_duration = robot_duration_hours.min()
total_robot_duration = robot_duration_hours.sum()
count_robot_encounters = len(robot_duration_hours)

# Since all robots have the same key (-1), you might want to just get overall statistics
# rather than grouping by the key
robot_providers_df.rename(columns = {'Encounter-Patient KEY' : 'DurableKey'}, inplace = True)

# df_exl = pd.read_excel('Heatmap22.xlsx')
# del df_exl['Unnamed: 0']
# df_exl.sample(frac = 0.02).to_excel('HeapMapZZZ.xlsx')
robot_providers_df

/var/folders/5y/g6ht6bhj6jn_xx5gg345nqhr0000gn/T/ipykernel_75961/3375410806.py:18: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,AdmissionInstant,DischargeInstant,DurableKey,Encounter-Provider KEY,Attending Provider KEY,Last Provider to Patient (KEY TO PROVIDER),Encounter - Department KEY,Encounter-Diagnosis KEY,Duration_Raw,Duration_Hours
Index,,,,,,,,,,
72680853,2022-02-24 12:19:26.666666746,2022-02-26 16:18:40.000000000,5862587,18397,-1,-1,10279,793876,2 days 03:59:13.333333254,51.987037
75349814,2022-03-11 15:52:53.333333254,2022-03-13 16:01:20.000000000,6820872,18692,-1,-1,10279,694326,2 days 00:08:26.666666746,48.140741
75868912,2022-03-26 19:26:20.000000000,2022-03-28 15:44:00.000000000,6223252,18692,-1,-1,10279,686369,1 days 20:17:40,44.294444
76348851,2022-04-10 22:59:46.666666746,2022-04-12 15:26:40.000000000,1453487,1103,-1,-1,349,847384,1 days 16:26:53.333333254,40.448148
76468188,2022-04-26 02:33:13.333333254,2022-04-27 15:09:20.000000000,7363,262619,-1,-1,10279,1016440,1 days 12:36:06.666666746,36.601852
...,...,...,...,...,...,...,...,...,...,...
152139065,2025-12-31 23:49:10.447761297,2025-12-31 23:49:11.194029808,5990465,303327,-1,-1,107,-1,0 days 00:00:00.746268511,0.000207
152139183,2025-12-31 23:51:08.358208895,2025-12-31 23:51:08.955223799,102954,442119,-1,-1,11824,-1,0 days 00:00:00.597014904,0.000166
152139319,2025-12-31 23:53:06.268656731,2025-12-31 23:53:06.716418028,6608114,2905,-1,-1,57,754800,0 days 00:00:00.447761297,0.000124


In [10]:
df_patients = pd.read_csv('patients.csv')
df_patients = df_patients.dropna()
df_patients = df_patients[df_patients['CensusBlockGroupFipsCode'] != '*Unspecified']
merged_df = pd.merge(df_patients, robot_providers_df, on='DurableKey', how = 'left')


In [11]:
# First, add the count to the merged DataFrame
merged_df['population_count'] = merged_df.groupby('CensusBlockGroupFipsCode')['CensusBlockGroupFipsCode'].transform('count')

# Calculate average Duration_Hours for each CensusBlockGroupFipsCode
avg_duration = merged_df.groupby('CensusBlockGroupFipsCode')['Duration_Hours'].mean().reset_index()
avg_duration.columns = ['CensusBlockGroupFipsCode', 'average_time']

# Create a new DataFrame with just location and count
location_counts_df = merged_df[['CensusBlockGroupFipsCode', 'population_count']].drop_duplicates()

# Merge the average duration into the location_counts_df
location_counts_df = location_counts_df.merge(avg_duration, on='CensusBlockGroupFipsCode', how='left')

# Sort by count in descending order (optional)
location_counts_df = location_counts_df.sort_values('population_count', ascending=False)

# Reset the index for cleaner output
location_counts_df = location_counts_df.reset_index(drop=True)

# Display the new DataFrame
print(location_counts_df)

     CensusBlockGroupFipsCode  population_count  average_time
0                201770036051             60376     12.603417
1                201770036042             60036     12.758295
2                201770034011             56944     12.779821
3                201770033022             54708     12.717963
4                201770039012             53166     12.688507
...                       ...               ...           ...
1310             201730101153                 2           NaN
1311             201379517001                 1           NaN
1312             390099729001                 1           NaN
1313             200099711003                 1           NaN
1314             200099718012                 1      2.255952

[1315 rows x 3 columns]


In [12]:
location_counts_df.dropna()

,CensusBlockGroupFipsCode,population_count,average_time
0,201770036051,60376,12.603417
1,201770036042,60036,12.758295
2,201770034011,56944,12.779821
3,201770033022,54708,12.717963
4,201770039012,53166,12.688507
...,...,...,...
1297,200910534233,4,38.585946
1302,201730103024,3,3.421277
1303,201939534001,2,27.454167
1309,201939531004,2,12.509314


In [ ]:
# First, make sure you have the required packages
# pip install kaleido imageio
# Shorten census block names
location_counts_df['ShortCode'] = location_counts_df['CensusBlockGroupFipsCode'].str[-4:]

# Create the 3D scatter plot with shortened names
fig = px.scatter_3d(location_counts_df, x='ShortCode', y='average_time', z='population_count',
                   color='average_time')

# Improve the layout
fig.update_layout(
    scene=dict(
        xaxis_title='Census Block (shortened)',
        yaxis_title='Average Time',
        zaxis_title='Population Count'
    ),
    title='Average Time by Robots in Counties'
)

# Create a temporary directory to store frames
import tempfile
temp_dir = tempfile.mkdtemp()
frames = []

# Generate frames by rotating the camera
for i, angle in enumerate(np.linspace(0, 2*np.pi, 60)):
    # Calculate camera position for rotation around z-axis
    camera_x = 1.5 * np.cos(angle)
    camera_y = 1.5 * np.sin(angle)
    camera_z = 0.8
    
    # Update camera position
    fig.update_layout(scene_camera=dict(eye=dict(x=camera_x, y=camera_y, z=camera_z)))
    
    # Save the frame as an image
    frame_path = os.path.join(temp_dir, f"frame_{i:03d}.png")
    fig.write_image(frame_path, width=800, height=600)
    frames.append(frame_path)

# Create a GIF from the frames
images = [imageio.imread(frame) for frame in frames]
gif_path = "3d_scatter.gif"
imageio.mimsave(gif_path, images, duration=0.1, loop=0)  # Added loop=0 for infinite looping

print(f"GIF saved as {gif_path}")

# Clean up temporary files
for frame in frames:
    os.remove(frame)
os.rmdir(temp_dir)

2026-03-29 11:46:26,030 - INFO     - Chromium init'ed with kwargs {}
2026-03-29 11:46:26,032 - INFO     - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-03-29 11:46:26,032 - INFO     - Temp directory created: /var/folders/5y/g6ht6bhj6jn_xx5gg345nqhr0000gn/T/tmpinffr122.
2026-03-29 11:46:26,034 - INFO     - Opening browser.
2026-03-29 11:46:26,035 - INFO     - Temp directory created: /var/folders/5y/g6ht6bhj6jn_xx5gg345nqhr0000gn/T/tmpiuqvacq2.
2026-03-29 11:46:26,035 - INFO     - Temporary directory at: /var/folders/5y/g6ht6bhj6jn_xx5gg345nqhr0000gn/T/tmpiuqvacq2
2026-03-29 11:46:27,783 - INFO     - Conforming 1 to file:///var/folders/5y/g6ht6bhj6jn_xx5gg345nqhr0000gn/T/tmpinffr122/index.html
2026-03-29 11:46:27,789 - INFO     - Waiting on all navigates
2026-03-29 11:46:28,554 - INFO     - All navigates done, putting them all in queue.
2026-03-29 11:46:28,555 - INFO     - Getting tab from queue (has 1)
2026-03-29 11:46:28,555 - INFO     - Got C88